# ML Modelling: EV Charging Network Optimisation
## IE Sustainability Datathon March 2026 | Iberdrola Challenge

This notebook trains the ML model pipeline that drives the three mandatory output files.
The feature matrix produced by `FE_v2.ipynb` is the direct input; all outputs here are
the primary submission artefacts.

**Mandatory Fork Reference (datos.gob.es):**
EV fleet projections (`total_ev_projected_2027 = 327,883`) are sourced from the official
datos.gob.es GitHub repository fork: **https://github.com/Jvilpi/Laboratorio-de-Datos**
(Exercise 3 — Open Data output, logistic penetration model).

| Section | Topic | Output |
|---|---|---|
| 1 | Setup | Imports, paths, constants |
| 2 | Load data | Feature matrix (494 × 62) |
| 3 | Feature sets | X_clf (59 cols) · X_reg (52 cols) |
| 4 | XGBoost Classifier | Which roads need stations? · SHAP |
| 5 | Two-Stage Regressor | How many stations? · SHAP |
| 6 | Road-level predictions | 494-road summary table |
| 7 | Placement logic | Station lat/lon along flagged roads |
| 8 | Grid status | Endesa KD-tree lookup + i-DE/Viesgo fallback |
| 9 | Output files | File 1, File 2, File 3 |
| 10 | BI Visualisation | Self-contained Folium HTML map |
| 11 | Verification | T5 compliance checks + printed schemas |

In [ ]:
# Uncomment when running in Google Colab
# !pip install xgboost shap geopandas folium optuna -q

# Google Drive mount for Colab (adjust path as needed)
# from google.colab import drive
# drive.mount('/content/drive')
# BASE = '/content/drive/MyDrive/Sustainability challenge'
# (then replace DATA_DIR below with BASE + '/Data')

In [ ]:
import os, json, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import geopandas as gpd
import shap
import xgboost as xgb
import folium
from scipy.spatial import cKDTree
from shapely.ops import unary_union, linemerge
from shapely.geometry import Point, MultiLineString
from sklearn.model_selection import GroupKFold
from sklearn.metrics import (
    f1_score, average_precision_score,
    precision_recall_curve, classification_report,
)

warnings.filterwarnings('ignore')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 200)

print('Imports OK')
print(f'XGBoost {xgb.__version__} | SHAP {shap.__version__}')

In [ ]:
# ── Paths (relative to notebooks/) ────────────────────────────────────────────
FM_PATH        = '../Data/processed/feature_matrix.csv'
V2_PATH        = '../Data/processed/ev_charging_ml_dataset_v2.csv'
RTIG_PATH      = '../Data/raw/road_routes_spain/carreteras_RTIG.geojson'
M2_PATH        = '../Data/interim/m2_charging_sites_interurban.csv'
M3_PATH        = '../Data/processed/m3_ev_projection.json'
ENDESA_RAW     = '../Data/external/grid_capacity_endesa/endesa_demanda_2026_03.csv'
ENDESA_INTERIM = '../Data/interim/m4_endesa_grid_capacity.csv'
OUT_DIR        = '../Data/processed'
os.makedirs(OUT_DIR, exist_ok=True)

# ── Datathon fixed constants ───────────────────────────────────────────────────
KW_PER_CHARGER       = 150      # Rule 2 — non-negotiable
EV_FLEET_2027        = 327_883  # M3 logistic projection (datos.gob.es fork)
AFIR_DENSITY         = 1.67     # stations / 100 km (EU 2023/1804 Art.4)
MAX_SPACING_KM       = 60       # AFIR = ≤60 km between fast chargers
MIN_CHARGERS         = 4
MAX_CHARGERS         = 12

# ── Grid thresholds (documented in Analytical Report) ─────────────────────────
THRESHOLD_SUFFICIENT = 10.0     # MW — ample for multi-hub HPC site
THRESHOLD_MODERATE   =  1.5     # MW — minimum for 10-charger × 150 kW hub

# ── Leakage discipline (inherited from FE_v2) ─────────────────────────────────
LEAKAGE_COLS = {'needs_more_stations', 'stations_per_100km', 'groupkfold_key'}
META_COLS    = ['carretera_group', 'y_clf', 'y_reg']

# Additional exclusions for regression model only
REGRESSION_EXTRA_EXCLUSIONS = [
    'n_stations', 'total_length_m', 'afir_stations_required',
    'log_n_stations', 'log_total_length_m', 'station_density_per_segment',
    'demand_vs_supply', 'refill_per_station',
    'ev_demand_2027_road', 'log_ev_demand_2027_road', 'demand_gap_score',
]

# Traffic weight → charger count mapping
TRAFFIC_WEIGHT = {
    'Autopista libre\\Autovía': 4.0,
    'Autopista peaje': 3.0,
    'Multicarril': 1.5,
    'Carretera convencional': 1.0,
}
CHARGERS_BY_TW = {4.0: 12, 3.0: 10, 1.5: 6, 1.0: 4}

print('Constants set:')
print(f'  KW_PER_CHARGER     = {KW_PER_CHARGER}')
print(f'  EV_FLEET_2027      = {EV_FLEET_2027:,}')
print(f'  AFIR spacing       = {MAX_SPACING_KM} km')
print(f'  Grid thresholds    = Sufficient ≥{THRESHOLD_SUFFICIENT} MW | Moderate ≥{THRESHOLD_MODERATE} MW')

---
## 2 · Load Feature Matrix

In [ ]:
assert os.path.exists(FM_PATH), f'Feature matrix not found: {FM_PATH}\nRun FE_v2.ipynb first.'

fm = pd.read_csv(FM_PATH)
print(f'Feature matrix loaded: {fm.shape}')
print(f'Columns: {list(fm.columns)}')

In [ ]:
# Separate features, targets, and metadata
groups    = fm['carretera_group'].copy()   # GroupKFold key (road names)
y_clf     = fm['y_clf'].copy()             # classification target
y_reg     = fm['y_reg'].copy()             # regression target
X_all     = fm.drop(columns=META_COLS)     # 59 feature columns

print(f'X_all shape    : {X_all.shape}')
print(f'Unique roads   : {groups.nunique()} (GroupKFold groups)')
print(f'NaN in X_all   : {X_all.isna().sum().sum()}')
print()
print('y_clf (needs_more_stations):')
print(y_clf.value_counts(normalize=True).round(3))
n_neg = int((y_clf == 0).sum())
n_pos = int((y_clf == 1).sum())
print(f'  Positives: {n_pos} / {len(y_clf)}  ({100*n_pos/len(y_clf):.1f}%)')
print(f'  scale_pos_weight = {n_neg/n_pos:.2f}')
print()
print('y_reg (AFIR station deficit):')
print(y_reg.describe())
print(f'  Non-zero: {(y_reg > 0).sum()} / {len(y_reg)}')

---
## 3 · Feature Sets

Two distinct feature matrices are built from `X_all`:

- **`X_clf`** (59 cols): All features. Used for the binary classifier (which roads need stations?).
- **`X_reg`** (52 cols): `X_clf` minus `REGRESSION_EXTRA_EXCLUSIONS`. Used for the regressor
  (how many stations?). The 7 excluded columns are derived from `n_stations` or `total_length_m`,
  which are the direct inputs to `y_reg = max(0, ceil(km × 1.67/100) − n_stations)`. Including
  them would let the model memorise the AFIR formula rather than learning from road attributes.

See `FE_v2.ipynb` Section 3.2 for the full leakage audit.

In [ ]:
# Classification feature set: all 59 features
X_clf = X_all.copy()

# Regression feature set: exclude leakage columns (only those present in X_all)
reg_excl_present = [c for c in REGRESSION_EXTRA_EXCLUSIONS if c in X_clf.columns]
X_reg = X_clf.drop(columns=reg_excl_present)

print(f'X_clf : {X_clf.shape[1]} features')
print(f'X_reg : {X_reg.shape[1]} features  (dropped: {reg_excl_present})')

# Snapshot: which features were actually excluded
not_found = [c for c in REGRESSION_EXTRA_EXCLUSIONS if c not in X_clf.columns]
if not_found:
    print(f'  (Already pruned in FE_v2, not in feature matrix: {not_found})')

---
## 4 · XGBoost Classifier

**Question**: *Which roads currently fall below the AFIR density threshold of 1.67 stations/100 km and need new charging infrastructure?*

**Validation strategy**: `GroupKFold(k=5)` on `carretera_group`. Roads in the same group
never split across train/val folds, preventing geographic data leakage from correlated segments.

**Metrics**: F1 (primary — penalises both missed positive roads and false alarms) +
PR-AUC (captures classifier ranking quality on the imbalanced 38% positive dataset).

In [ ]:
gkf = GroupKFold(n_splits=5)
scale_pos = n_neg / n_pos

clf_params = dict(
    n_estimators=400,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos,
    eval_metric='aucpr',
    use_label_encoder=False,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

cv_results = []
oof_proba  = np.zeros(len(X_clf))   # out-of-fold probabilities for threshold tuning

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_clf, y_clf, groups=groups)):
    X_tr,  X_val  = X_clf.iloc[tr_idx],  X_clf.iloc[val_idx]
    y_tr,  y_val  = y_clf.iloc[tr_idx],  y_clf.iloc[val_idx]

    model = xgb.XGBClassifier(**clf_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    proba = model.predict_proba(X_val)[:, 1]
    pred  = (proba >= 0.5).astype(int)

    oof_proba[val_idx] = proba
    f1    = f1_score(y_val, pred)
    prauc = average_precision_score(y_val, proba)
    cv_results.append({'fold': fold+1, 'f1': f1, 'pr_auc': prauc,
                       'n_val': len(val_idx), 'n_pos_val': int(y_val.sum())})
    print(f'  Fold {fold+1}: F1={f1:.3f}  PR-AUC={prauc:.3f}  '
          f'(val_n={len(val_idx)}, val_pos={int(y_val.sum())})')

cv_df = pd.DataFrame(cv_results)
print(f'\nMean F1    : {cv_df["f1"].mean():.3f} ± {cv_df["f1"].std():.3f}')
print(f'Mean PR-AUC: {cv_df["pr_auc"].mean():.3f} ± {cv_df["pr_auc"].std():.3f}')

In [ ]:
# Find the probability threshold that maximises OOF F1
thresholds = np.arange(0.05, 0.95, 0.01)
f1_by_t    = [f1_score(y_clf, (oof_proba >= t).astype(int)) for t in thresholds]

best_thresh = float(thresholds[np.argmax(f1_by_t)])
best_f1_oof = float(np.max(f1_by_t))

print(f'Optimal threshold (OOF F1): {best_thresh:.2f}  →  F1={best_f1_oof:.3f}')
print(f'Default threshold   (0.50): F1={f1_score(y_clf, (oof_proba >= 0.50).astype(int)):.3f}')

# Precision-Recall curve
precision, recall, pr_thresholds = precision_recall_curve(y_clf, oof_proba)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(recall, precision, color='steelblue', lw=2)
axes[0].set_xlabel('Recall'); axes[0].set_ylabel('Precision')
axes[0].set_title(f'Precision-Recall Curve (OOF)  |  PR-AUC={average_precision_score(y_clf, oof_proba):.3f}')
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, f1_by_t, color='darkorange', lw=2)
axes[1].axvline(best_thresh, color='red', linestyle='--', label=f'Best = {best_thresh:.2f}')
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('F1')
axes[1].set_title('F1 vs Classification Threshold (OOF)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# OOF classification report at best threshold
y_oof_pred = (oof_proba >= best_thresh).astype(int)
print('\nOOF Classification Report (at best threshold):')
print(classification_report(y_clf, y_oof_pred, target_names=['No need', 'Needs stations']))

In [ ]:
# Final classifier: train on full dataset (all 494 roads)
clf_final = xgb.XGBClassifier(**clf_params)
clf_final.fit(X_clf, y_clf, verbose=False)

print('Final classifier trained on 494 roads.')
print(f'Best threshold (from OOF): {best_thresh:.2f}')

### 4.1 SHAP Feature Importance — Classifier

SHAP (SHapley Additive exPlanations) decomposes each prediction into additive
feature contributions. For the datathon presentation, the top drivers explain
*why* certain roads are flagged for new charging infrastructure.

In [ ]:
explainer_clf  = shap.TreeExplainer(clf_final)
shap_vals_clf  = explainer_clf.shap_values(X_clf)   # shape (494, 59)

# Global bar chart (mean |SHAP|)
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.sca(axes[0])
shap.summary_plot(shap_vals_clf, X_clf, plot_type='bar', max_display=20, show=False)
axes[0].set_title('Mean |SHAP| — Classifier (top 20 features)')

plt.sca(axes[1])
shap.summary_plot(shap_vals_clf, X_clf, plot_type='dot', max_display=20, show=False)
axes[1].set_title('SHAP Value Distribution — Classifier')
plt.tight_layout()
plt.show()

# Print top 10 features by mean |SHAP|
mean_abs_shap_clf = pd.Series(
    np.abs(shap_vals_clf).mean(axis=0), index=X_clf.columns
).sort_values(ascending=False)
print('Top 10 features (mean |SHAP|) — Classifier:')
print(mean_abs_shap_clf.head(10).round(4).to_string())

---
## 5 · Two-Stage Regressor

**Question**: *For roads that need more stations, how many stations should be added to
meet the AFIR 1.67/100 km density threshold?*

The target `y_reg = max(0, ⌈road_km × 1.67/100⌉ − n_existing_stations)` is zero-inflated:
62% of roads already meet AFIR. The two-stage approach handles this cleanly:
- **Stage 1** — Classifier (Section 4): decides whether deficit > 0
- **Stage 2** — Regressor (this section): trained *only on positive roads* (187 rows),
  estimates the size of the deficit

Using `objective='count:poisson'` is natural for integer count targets with a lower bound of 1.

In [ ]:
from sklearn.metrics import mean_absolute_error

# Restrict to positive roads (y_clf == 1; y_reg guaranteed ≥ 1 here)
pos_mask   = (y_clf == 1).values
X_reg_pos  = X_reg.loc[pos_mask].copy()
y_reg_pos  = y_reg.loc[pos_mask].copy()
groups_pos = groups.loc[pos_mask].copy()

print(f'Positive subset: {pos_mask.sum()} roads | y_reg range: {y_reg_pos.min()}–{y_reg_pos.max()}')

reg_params = dict(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.7,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    objective='count:poisson',
    eval_metric='mae',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

reg_cv_results = []
oof_reg        = np.zeros(pos_mask.sum())

for fold, (tr_idx, val_idx) in enumerate(gkf.split(X_reg_pos, y_reg_pos, groups=groups_pos)):
    X_tr,  X_val  = X_reg_pos.iloc[tr_idx],  X_reg_pos.iloc[val_idx]
    y_tr,  y_val  = y_reg_pos.iloc[tr_idx],  y_reg_pos.iloc[val_idx]

    model = xgb.XGBRegressor(**reg_params)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)

    pred = np.maximum(1, np.round(model.predict(X_val))).astype(int)
    mae  = mean_absolute_error(y_val, pred)
    oof_reg[val_idx] = pred
    reg_cv_results.append({'fold': fold+1, 'mae': mae, 'n_val': len(val_idx)})
    print(f'  Fold {fold+1}: MAE={mae:.3f}  (val_n={len(val_idx)})')

reg_cv_df = pd.DataFrame(reg_cv_results)
print(f'\nMean MAE: {reg_cv_df["mae"].mean():.3f} ± {reg_cv_df["mae"].std():.3f}')
print(f'Baseline MAE (predict mean={y_reg_pos.mean():.2f}): '
      f'{mean_absolute_error(y_reg_pos, np.full(len(y_reg_pos), round(y_reg_pos.mean()))):.3f}')

In [ ]:
# Final regressor: trained on all 187 positive roads
reg_final = xgb.XGBRegressor(**reg_params)
reg_final.fit(X_reg_pos, y_reg_pos, verbose=False)

# OOF prediction distribution
print(f'Regressor trained on {len(X_reg_pos)} positive roads.')
print(f'OOF y_reg predictions:')
oof_int = np.maximum(1, np.round(oof_reg)).astype(int)
print(pd.Series(oof_int).value_counts().sort_index().to_string())

### 5.1 SHAP Feature Importance — Regressor

In [ ]:
explainer_reg = shap.TreeExplainer(reg_final)
shap_vals_reg = explainer_reg.shap_values(X_reg_pos)   # shape (187, 52)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
plt.sca(axes[0])
shap.summary_plot(shap_vals_reg, X_reg_pos, plot_type='bar', max_display=15, show=False)
axes[0].set_title('Mean |SHAP| — Regressor (top 15 features)')

plt.sca(axes[1])
shap.summary_plot(shap_vals_reg, X_reg_pos, plot_type='dot', max_display=15, show=False)
axes[1].set_title('SHAP Value Distribution — Regressor')
plt.tight_layout()
plt.show()

mean_abs_shap_reg = pd.Series(
    np.abs(shap_vals_reg).mean(axis=0), index=X_reg_pos.columns
).sort_values(ascending=False)
print('Top 10 features (mean |SHAP|) — Regressor:')
print(mean_abs_shap_reg.head(10).round(4).to_string())

---
## 6 · Road-Level Predictions

Apply both models to all 494 roads to generate the final prediction table.
Roads where `needs_stations=1` will receive new charging stations in Section 7.

In [ ]:
# Stage 1: Classify all 494 roads
clf_proba = clf_final.predict_proba(X_clf)[:, 1]
clf_pred  = (clf_proba >= best_thresh).astype(int)

# Stage 2: Predict deficit only for clf=1 roads
reg_pred = np.zeros(len(X_clf), dtype=int)
clf1_mask = clf_pred == 1
if clf1_mask.sum() > 0:
    raw = reg_final.predict(X_reg.loc[clf1_mask])
    reg_pred[clf1_mask] = np.maximum(1, np.round(raw)).astype(int)

# Road-level prediction table
road_preds = pd.DataFrame({
    'carretera_group'   : groups.values,
    'clf_proba'         : clf_proba.round(3),
    'needs_stations'    : clf_pred,
    'n_new_stations'    : reg_pred,
    'y_clf_true'        : y_clf.values,
    'y_reg_true'        : y_reg.values,
    'traffic_weight'    : X_clf['traffic_weight'].values,
    'grid_data_missing' : fm.get('comunidad_autonoma_unknown',
                                  pd.Series(np.zeros(len(fm)), index=fm.index)).values,
})

print('Road predictions summary:')
print(f'  Roads flagged by classifier  : {clf_pred.sum()} / {len(clf_pred)}')
print(f'  Total new stations predicted : {reg_pred.sum()}')
print(f'  y_clf true positives         : {y_clf.sum()}')
print()
print('New stations per road (predicted):')
print(pd.Series(reg_pred[clf1_mask]).value_counts().sort_index().to_string())
print()
print(road_preds[road_preds['needs_stations']==1]
      [['carretera_group','clf_proba','n_new_stations','traffic_weight']]
      .sort_values('clf_proba', ascending=False).head(20).to_string(index=False))

---
## 7 · Placement Logic — Station Coordinates

For each road flagged by the classifier, `n_new_stations` charging stations are placed at
equal spacing along the road geometry. The spacing formula is:

$$\text{interval} = \frac{L}{N_{new}+1}$$

where $L$ is the road's total length (metres, EPSG:25830) and $N_{new}$ is the predicted
station deficit. This guarantees roughly equal coverage along the road.

**Assumption documented for Analytical Report:**
Station positions are interpolated along the centreline of the road geometry from the
Ministry of Transport RTIG dataset. Real-world siting would adjust for service area
availability, land access, and local grid connection feasibility.

**n_chargers formula**: `n_chargers = CHARGERS_BY_TW[traffic_weight]`:
Autopista/Autovía (TW=4) → 12 chargers | Peaje (TW=3) → 10 | Multicarril (TW=1.5) → 6 | Convencional (TW=1) → 4.

In [ ]:
assert os.path.exists(RTIG_PATH), f'RTIG geojson not found: {RTIG_PATH}'

roads_gdf = gpd.read_file(RTIG_PATH)
print(f'RTIG loaded: {len(roads_gdf):,} segments | CRS: {roads_gdf.crs}')
print(f'Road name field: "Carretera" — sample: {roads_gdf["Carretera"].dropna().head(3).tolist()}')

# Group segments by road name for geometry merging
roads_by_name = roads_gdf.groupby('Carretera')
geojson_roads = set(roads_gdf['Carretera'].dropna().unique())
print(f'Unique roads in RTIG: {len(geojson_roads)}')

# Roads flagged but not found in geojson (should be zero — log for Analytical Report)
flagged_roads = road_preds[road_preds['needs_stations'] == 1]['carretera_group'].tolist()
missing_geom  = [r for r in flagged_roads if r not in geojson_roads]
print(f'Flagged roads without geometry: {len(missing_geom)} → {missing_geom[:10]}')

In [ ]:
METRIC_CRS = 'EPSG:25830'


def merge_road_geometry(road_name: str) -> object:
    """Return a single (Multi)LineString for a road in METRIC_CRS."""
    segs = roads_by_name.get_group(road_name).to_crs(METRIC_CRS)
    combined = segs.geometry.unary_union
    merged   = linemerge(combined)
    if isinstance(merged, MultiLineString):
        # Non-contiguous segments — use the longest component
        merged = max(merged.geoms, key=lambda g: g.length)
    return merged


def place_stations(geom, n_new: int) -> list:
    """Interpolate n_new points at equal spacing along a LineString."""
    L = geom.length
    return [geom.interpolate((i + 1) * L / (n_new + 1)) for i in range(n_new)]


station_rows = []
skipped      = []

for _, row in road_preds[road_preds['needs_stations'] == 1].iterrows():
    road_name = row['carretera_group']
    n_new     = int(row['n_new_stations'])
    tw        = float(row['traffic_weight'])

    if road_name not in geojson_roads or n_new <= 0:
        skipped.append(road_name)
        continue

    try:
        geom_m    = merge_road_geometry(road_name)
        points_m  = place_stations(geom_m, n_new)
    except Exception as e:
        skipped.append(f'{road_name} ({e})')
        continue

    # Convert points back to WGS84
    pts_wgs84 = gpd.GeoSeries(points_m, crs=METRIC_CRS).to_crs('EPSG:4326')

    # n_chargers from traffic weight (documented assumption)
    n_chargers = CHARGERS_BY_TW.get(tw, MIN_CHARGERS)

    for pt in pts_wgs84:
        station_rows.append({
            'route_segment'      : road_name,
            'latitude'           : round(pt.y, 6),
            'longitude'          : round(pt.x, 6),
            'n_chargers_proposed': n_chargers,
        })

df_stations = pd.DataFrame(station_rows)
print(f'Station rows generated: {len(df_stations)}')
if skipped:
    print(f'Skipped roads (no geometry or n_new=0): {skipped}')
print(df_stations.head(5).to_string(index=False))

---
## 8 · Grid Status Assignment

Each proposed station is matched to its nearest electrical substation to determine
whether the local grid can support a 150 kW HPC hub.

**Spatial matching methodology (Rule 1):** KD-tree nearest-neighbour search on WGS84
decimal degree coordinates. The nearest substation's `available_mw` determines `grid_status`
using the thresholds: Sufficient ≥10 MW · Moderate 1.5–10 MW · Congested <1.5 MW.

**Data availability:**
- **Endesa**: raw CSV loaded inline below → 1,034 substations
- **i-DE (Iberdrola)**: data not yet downloaded → fallback: `Moderate` status, distributor
  assigned from region mapping (documented assumption)
- **Viesgo**: data not yet downloaded → same fallback

This is a documented limitation. All roads with `comunidad_autonoma_unknown=1` (i-DE/Viesgo
territory, 176 roads) receive the conservative `Moderate` fallback.

In [ ]:
# ── Inline Endesa Grid Prep (ported from M6 / R2 inline cells) ─────────────────
# Reuses exact same logic as Network_Optimisation.ipynb cell 101a9278

if os.path.exists(ENDESA_INTERIM):
    df_endesa = pd.read_csv(ENDESA_INTERIM)
    print(f'Loaded cached Endesa interim: {len(df_endesa):,} substations')
else:
    assert os.path.exists(ENDESA_RAW), f'Endesa raw CSV not found: {ENDESA_RAW}'
    import math as _math

    _df = pd.read_csv(ENDESA_RAW, sep=';', encoding='latin-1')
    _df.columns = _df.columns.str.strip()

    def _parse_es(col):
        return col.astype(str).str.replace(',', '.', regex=False).astype(float)

    _df['utm_x']    = _parse_es(_df['Coordenada UTM X'])
    _df['utm_y']    = _parse_es(_df['Coordenada UTM Y'])
    _df['avail_mw'] = _parse_es(_df['Capacidad firme disponible (MW)'])
    _df['volt_kv']  = _parse_es(_df.iloc[:, 6])
    _df = _df[(_df['utm_x'] > 0) & (_df['utm_y'] > 0)].copy()

    _agg = _df.groupby(['utm_x', 'utm_y'], as_index=False).agg(
        available_mw   = ('avail_mw', 'sum'),
        max_voltage_kv = ('volt_kv',  'max')
    )

    def _utm_to_latlon(easting, northing, zone=30):
        a, f = 6378137.0, 1 / 298.257223563
        b = a*(1-f); e2 = 1-(b/a)**2; e_p2 = e2/(1-e2)
        k0 = 0.9996; E0 = 500000
        x = easting - E0; y = northing
        M = y / k0
        mu = M / (a*(1-e2/4-3*e2**2/64-5*e2**3/256))
        e1 = (1-_math.sqrt(1-e2))/(1+_math.sqrt(1-e2))
        phi1 = (mu
                + (3*e1/2-27*e1**3/32)*_math.sin(2*mu)
                + (21*e1**2/16-55*e1**4/32)*_math.sin(4*mu)
                + (151*e1**3/96)*_math.sin(6*mu))
        N1 = a/_math.sqrt(1-e2*_math.sin(phi1)**2)
        T1 = _math.tan(phi1)**2; C1 = e_p2*_math.cos(phi1)**2
        R1 = a*(1-e2)/(1-e2*_math.sin(phi1)**2)**1.5
        D = x/(N1*k0)
        lat = phi1 - (N1*_math.tan(phi1)/R1)*(
            D**2/2-(5+3*T1+10*C1-4*C1**2-9*e_p2)*D**4/24
            +(61+90*T1+298*C1+45*T1**2-252*e_p2-3*C1**2)*D**6/720)
        lon_r = (D-(1+2*T1+C1)*D**3/6
                 +(5-2*C1+28*T1-3*C1**2+8*e_p2+24*T1**2)*D**5/120
                 )/_math.cos(phi1)
        return _math.degrees(lat), _math.degrees(lon_r)+(6*zone-183)

    _lats, _lons = zip(*[_utm_to_latlon(r.utm_x, r.utm_y) for r in _agg.itertuples()])
    _agg['latitude']  = _lats
    _agg['longitude'] = _lons
    _agg['grid_status'] = np.select(
        [_agg['available_mw'] >= THRESHOLD_SUFFICIENT,
         _agg['available_mw'] >= THRESHOLD_MODERATE],
        ['Sufficient', 'Moderate'], default='Congested'
    )

    df_endesa = _agg[['latitude','longitude','available_mw','grid_status','max_voltage_kv']]
    df_endesa.to_csv(ENDESA_INTERIM, index=False)
    print(f'Endesa prep complete: {len(df_endesa):,} substations → {ENDESA_INTERIM}')

df_endesa['distributor_network'] = 'Endesa'
print(df_endesa['grid_status'].value_counts().to_string())

In [ ]:
# ── Distributor territory mapping (for i-DE / Viesgo fallback) ────────────────
# Source: Spanish CNMC regulated distributor territory assignments
REGION_TO_DISTRIBUTOR = {
    '01 - Andalucía'   : 'Endesa',   '04 - Illes Balears': 'Endesa',
    '05 - Canarias'    : 'Endesa',   '09 - Catalunya'    : 'Endesa',
    '11 - Extremadura' : 'Endesa',   '02 - Aragón'       : 'Endesa',
    '03 - Asturias'    : 'Viesgo',   '06 - Cantabria'    : 'Viesgo',
}  # All other regions default to i-DE (Iberdrola)

# ── Build Endesa KD-tree ───────────────────────────────────────────────────────
endesa_coords = df_endesa[['latitude', 'longitude']].values
endesa_tree   = cKDTree(endesa_coords)

# ── Load v2 dataset to get comunidad_autonoma per road ────────────────────────
# Note: v2 CSV uses original column names (Carretera, comunidad_autonoma)
df_v2_meta = pd.read_csv(V2_PATH)[['Carretera', 'comunidad_autonoma']].copy()
df_v2_meta.columns = ['carretera_group', 'comunidad_autonoma']


def lookup_grid_status(lat: float, lon: float,
                        road_name: str,
                        comunidad: str) -> dict:
    """
    Return grid_status, available_mw, and distributor_network for a proposed station.
    Roads in Endesa territory: nearest substation from KD-tree.
    Roads in i-DE/Viesgo territory (no data): conservative Moderate fallback.
    Documented assumption: absence of distributor data → Moderate (cannot confirm capacity).
    """
    distributor = REGION_TO_DISTRIBUTOR.get(str(comunidad), 'i-DE')

    if distributor == 'Endesa':
        dist, idx = endesa_tree.query([lat, lon])
        sub_row   = df_endesa.iloc[idx]
        return {
            'grid_status'        : sub_row['grid_status'],
            'available_mw'       : float(sub_row['available_mw']),
            'distributor_network': 'Endesa',
        }
    else:
        # i-DE or Viesgo: no capacity data → Moderate (conservative planning assumption)
        return {
            'grid_status'        : 'Moderate',
            'available_mw'       : np.nan,
            'distributor_network': distributor,
        }


# ── Assign grid status to every proposed station ──────────────────────────────
road_region_map = df_v2_meta.set_index('carretera_group')['comunidad_autonoma'].to_dict()

grid_cols = df_stations.apply(
    lambda row: pd.Series(lookup_grid_status(
        row['latitude'], row['longitude'],
        row['route_segment'],
        road_region_map.get(row['route_segment'], 'unknown')
    )),
    axis=1
)
df_stations = pd.concat([df_stations, grid_cols], axis=1)

print('Grid status assigned to all proposed stations:')
print(df_stations['grid_status'].value_counts().to_string())
print()
print('Distributor breakdown:')
print(df_stations['distributor_network'].value_counts().to_string())

---
## 9 · Build Output Files

Generate the three mandatory CSV files exactly as specified in the datathon guidelines.
All field names, data types, and standardisation rules are enforced by assertions.

In [ ]:
# ── FILE 2: Proposed Charging Locations ───────────────────────────────────────
file2 = pd.DataFrame({
    'location_id'        : [f'IBE_{str(i+1).zfill(3)}' for i in range(len(df_stations))],
    'latitude'           : df_stations['latitude'].values,
    'longitude'          : df_stations['longitude'].values,
    'route_segment'      : df_stations['route_segment'].values,
    'n_chargers_proposed': df_stations['n_chargers_proposed'].astype(int).values,
    'grid_status'        : df_stations['grid_status'].values,
})

# Validation
valid_statuses = {'Sufficient', 'Moderate', 'Congested'}
assert set(file2['grid_status'].unique()).issubset(valid_statuses), \
    f'Invalid grid_status: {set(file2["grid_status"].unique()) - valid_statuses}'
assert file2['latitude'].between(27, 45).all(),  'Latitude out of Spain bounds'
assert file2['longitude'].between(-20, 5).all(), 'Longitude out of Spain bounds'
assert file2['n_chargers_proposed'].between(MIN_CHARGERS, MAX_CHARGERS).all(), \
    'n_chargers outside expected range'

FILE2_PATH = os.path.join(OUT_DIR, 'File_2.csv')
file2.to_csv(FILE2_PATH, index=False)
print(f'File 2: {len(file2):,} rows → {FILE2_PATH}')
print(file2.head(5).to_string(index=False))

In [ ]:
# ── FILE 3: Friction Points ─────────────────────────────────────────────────────
# Only Moderate or Congested locations from File 2 (Rule 3)
f3_mask    = file2['grid_status'].isin(['Moderate', 'Congested'])
f3_source  = file2[f3_mask].copy()
f3_distrib = df_stations.loc[f3_mask.values, 'distributor_network'].values

file3 = pd.DataFrame({
    'bottleneck_id'      : [f'FRIC_{str(i+1).zfill(3)}' for i in range(len(f3_source))],
    'latitude'           : f3_source['latitude'].values,
    'longitude'          : f3_source['longitude'].values,
    'route_segment'      : f3_source['route_segment'].values,
    'distributor_network': f3_distrib,
    'estimated_demand_kw': (f3_source['n_chargers_proposed'].values * KW_PER_CHARGER).astype(float),
    'grid_status'        : f3_source['grid_status'].values,
})

# Validation (Rules 2 & 3)
assert 'Sufficient' not in file3['grid_status'].values, \
    'Rule 3 violation: Sufficient locations in File 3'
assert (file3['estimated_demand_kw'] == file3.apply(
    lambda r: r['estimated_demand_kw'] // KW_PER_CHARGER * KW_PER_CHARGER, axis=1)).all(), \
    'Rule 2 violation: estimated_demand_kw not a multiple of 150 kW'
valid_distributors = {'i-DE', 'Endesa', 'Viesgo'}
assert set(file3['distributor_network'].unique()).issubset(valid_distributors), \
    f'Invalid distributor: {set(file3["distributor_network"].unique()) - valid_distributors}'

FILE3_PATH = os.path.join(OUT_DIR, 'File_3.csv')
file3.to_csv(FILE3_PATH, index=False)
print(f'File 3: {len(file3):,} friction points → {FILE3_PATH}')
print(file3['grid_status'].value_counts().to_string())
print()
print(file3.head(5).to_string(index=False))

In [ ]:
# ── FILE 1: Global Network KPIs ────────────────────────────────────────────────
# Load M2 baseline and M3 projection for the required KPI fields
assert os.path.exists(M2_PATH), f'M2 output not found: {M2_PATH}'
assert os.path.exists(M3_PATH), f'M3 output not found: {M3_PATH}'

df_existing = pd.read_csv(M2_PATH)
total_existing = int(df_existing['site_id'].nunique() if 'site_id' in df_existing.columns
                     else len(df_existing))

with open(M3_PATH) as _f:
    m3 = json.load(_f)
total_ev_2027 = int(m3['total_ev_projected_2027'])

file1 = pd.DataFrame([{
    'total_proposed_stations'         : int(len(file2)),
    'total_existing_stations_baseline': total_existing,
    'total_friction_points'           : int(len(file3)),
    'total_ev_projected_2027'         : total_ev_2027,
}])

# Validation
assert file1['total_proposed_stations'].iloc[0]  == len(file2)
assert file1['total_friction_points'].iloc[0]    == len(file3)
assert file1['total_ev_projected_2027'].iloc[0]  == EV_FLEET_2027

FILE1_PATH = os.path.join(OUT_DIR, 'File_1.csv')
file1.to_csv(FILE1_PATH, index=False)

print('FILE 1 — GLOBAL NETWORK KPIs')
print('=' * 55)
for col in file1.columns:
    print(f'  {col:<42}: {file1[col].iloc[0]:,}')
print('=' * 55)

---
## 10 · BI Visualisation

Self-contained Folium HTML map — opens directly in any browser with no installation
or login required (mandatory accessibility requirement, evaluation criterion B4).

Colour coding: **Green** = Sufficient | **Orange** = Moderate | **Red** = Congested.

In [ ]:
STATUS_COLORS = {'Sufficient': 'green', 'Moderate': 'orange', 'Congested': 'red'}

m = folium.Map(location=[40.4, -3.7], zoom_start=6, tiles='CartoDB positron')

for _, row in file2.iterrows():
    color = STATUS_COLORS.get(row['grid_status'], 'grey')
    folium.CircleMarker(
        location=[row['latitude'], row['longitude']],
        radius=7,
        color=color,
        fill=True,
        fill_color=color,
        fill_opacity=0.85,
        popup=folium.Popup(
            f"<b>{row['location_id']}</b><br>"
            f"Route: {row['route_segment']}<br>"
            f"Chargers: {row['n_chargers_proposed']}<br>"
            f"Demand: {row['n_chargers_proposed'] * KW_PER_CHARGER} kW<br>"
            f"Grid: <b>{row['grid_status']}</b>",
            max_width=200,
        ),
        tooltip=f"{row['location_id']} | {row['route_segment']} | {row['grid_status']}",
    ).add_to(m)

# Legend
legend_html = """
<div style='position:fixed; bottom:30px; left:30px; z-index:1000;
     background:white; padding:12px 16px; border-radius:8px;
     border:1px solid #ccc; font-size:13px; line-height:1.9'>
  <b>IE Datathon 2026 — Proposed EV Charging Stations</b><br>
  <span style='color:green'>&#9679;</span> Sufficient (&ge;10 MW)<br>
  <span style='color:orange'>&#9679;</span> Moderate (1.5–10 MW)<br>
  <span style='color:red'>&#9679;</span> Congested (&lt;1.5 MW)<br>
  <hr style='margin:6px 0'>
  <small>datos.gob.es fork: github.com/Jvilpi/Laboratorio-de-Datos</small>
</div>"""
m.get_root().html.add_child(folium.Element(legend_html))

MAP_PATH = os.path.join(OUT_DIR, 'BI_Visualization.html')
m.save(MAP_PATH)
size_kb = os.path.getsize(MAP_PATH) / 1024
print(f'BI map saved: {MAP_PATH}  ({size_kb:.0f} KB)')
print('Opens directly in any browser — no installation or login required.')

m   # display inline if running in Jupyter

---
## 11 · Output Verification (T5)

Per evaluation criterion **T5**, all output file structures must be printed with
visible outputs, and all compliance checks must pass.

In [ ]:
print('=' * 70)
print('  OUTPUT VERIFICATION — ALL THREE FILES')
print('=' * 70)

for label, path, df in [('FILE 1', FILE1_PATH, file1),
                          ('FILE 2', FILE2_PATH, file2),
                          ('FILE 3', FILE3_PATH, file3)]:
    df_v = pd.read_csv(path)
    print(f'\n--- {label} ---')
    print(f'  Path    : {path}')
    print(f'  Rows    : {len(df_v):,}')
    print(f'  Columns : {df_v.columns.tolist()}')
    print(f'  Dtypes  :')
    for col, dtype in df_v.dtypes.items():
        print(f'    {col:<42} {dtype}')
    print(f'  Sample  :')
    print(df_v.head(3).to_string(index=False))

print()
print('=' * 70)
print('  COMPLIANCE CHECKS')
print('=' * 70)

# Rule 1
assert set(file2['grid_status'].unique()).issubset({'Sufficient','Moderate','Congested'})
print('  ✓ Rule 1 : grid_status values valid (Sufficient / Moderate / Congested)')

# Rule 2
assert (file3['estimated_demand_kw'] % KW_PER_CHARGER == 0).all()
print(f'  ✓ Rule 2 : estimated_demand_kw = n_chargers × {KW_PER_CHARGER} kW (fixed)')

# Rule 3
assert 'Sufficient' not in file3['grid_status'].values
print('  ✓ Rule 3 : File 3 contains no Sufficient locations')

# File 1 consistency
assert file1['total_proposed_stations'].iloc[0]  == len(file2)
assert file1['total_friction_points'].iloc[0]    == len(file3)
assert file1['total_ev_projected_2027'].iloc[0]  == EV_FLEET_2027
print('  ✓ File 1 KPIs are consistent with File 2 and File 3 row counts')

# Coordinates
assert file2['latitude'].between(27, 45).all()
assert file2['longitude'].between(-20, 5).all()
print('  ✓ All coordinates within Spain bounding box (27–45°N, 20°W–5°E)')

# Distributor values
assert set(file3['distributor_network'].unique()).issubset({'i-DE','Endesa','Viesgo'})
print('  ✓ distributor_network values valid (i-DE / Endesa / Viesgo)')

# Location ID format
assert file2['location_id'].str.match(r'^IBE_\d{3}$').all()
assert file3['bottleneck_id'].str.match(r'^FRIC_\d{3}$').all()
print('  ✓ location_id format: IBE_XXX | bottleneck_id format: FRIC_XXX')

print()
print('  ALL COMPLIANCE CHECKS PASSED — files ready for submission.')

---
## 12 · Summary for Analytical Report

Copy these figures directly into the Analytical Report and Final Presentation.

In [ ]:
print('NETWORK OPTIMISATION — SUMMARY STATISTICS (ML Model)')
print('=' * 62)
print(f'  Total proposed new stations      : {len(file2):,}')
print(f'  Existing interurban baseline     : {total_existing:,}')
print(f'  Total friction points            : {len(file3):,}')
print(f'  Projected EV fleet 2027          : {EV_FLEET_2027:,}')
print()
print('Grid status breakdown (File 2):')
for status, count in file2['grid_status'].value_counts().items():
    print(f'  {status:<12} : {count:>4}  ({100*count/len(file2):.1f}%)')
print()
print('Chargers proposed:')
print(f'  Total chargers                   : {file2["n_chargers_proposed"].sum():,}')
print(f'  Avg per station                  : {file2["n_chargers_proposed"].mean():.1f}')
print(f'  Total installed capacity         : {file2["n_chargers_proposed"].sum() * KW_PER_CHARGER:,} kW')
print()
print('Model performance (GroupKFold k=5):')
print(f'  Classifier  — Mean F1    : {cv_df["f1"].mean():.3f} ± {cv_df["f1"].std():.3f}')
print(f'  Classifier  — Mean PR-AUC: {cv_df["pr_auc"].mean():.3f} ± {cv_df["pr_auc"].std():.3f}')
print(f'  Regressor   — Mean MAE   : {reg_cv_df["mae"].mean():.3f} ± {reg_cv_df["mae"].std():.3f}')
print(f'  Best clf threshold (OOF) : {best_thresh:.2f}')
print()
print('Key assumptions (reproduce in Analytical Report):')
print(f'  EV fleet 2027 (datos.gob.es fork): {EV_FLEET_2027:,} BEVs')
print(f'  AFIR density threshold            : {AFIR_DENSITY} stations / 100 km')
print(f'  Max station spacing               : {MAX_SPACING_KM} km')
print(f'  Standard power per charger        : {KW_PER_CHARGER} kW (fixed by rules)')
print(f'  Grid thresholds                   : Sufficient ≥{THRESHOLD_SUFFICIENT} MW | '
      f'Moderate ≥{THRESHOLD_MODERATE} MW | Congested <{THRESHOLD_MODERATE} MW')
print(f'  i-DE/Viesgo fallback              : Moderate (no capacity data available)')
print()
print('Mandatory fork reference:')
print('  https://github.com/Jvilpi/Laboratorio-de-Datos')
print('  (datos.gob.es Route to Electrification — Exercise 3 output)')